In [1]:
import numpy as np

In [3]:
"""
Normalized NFW profile: enclosed mass fraction within an inner radius,
as a function of concentration, integrated via Romberg in log space.
"""
import numpy as np
from scipy.integrate import romb


def nfw_shape(x):
    """Unnormalized NFW density shape rho(x) ~ 1/(x*(1+x)^2), x = r/r_s."""
    return 1.0 / (x * (1.0 + x)**2)


def m_of_c(c):
    """Analytic NFW mass profile shape function m(c) = ln(1+c) - c/(1+c)."""
    return np.log(1.0 + c) - c / (1.0 + c)


def enclosed_fraction_romberg(c, x_inner, x_min=1e-6, n_intervals_pow2=16):
    """
    Fraction of total (0 -> c, i.e. 0 -> r_vir) NFW mass enclosed within
    x_inner = r/r_s, computed by Romberg integration on a log-spaced grid.

    Integral of 4*pi*r^2*rho(r) dr  ->  in x = r/r_s, dr = r_s dx, so
    integrand ~ x^2 * rho_shape(x) dx. Substituting x = exp(u) (log-space,
    uniform grid), dx = x du, so the integrand becomes x^3 * rho_shape(x) du.

    Parameters
    ----------
    c : float
        Host/subhalo concentration (defines outer integration limit, x=c).
    x_inner : float
        Inner radius in units of r_s (r/r_s) defining "inner region".
    x_min : float
        Lower cutoff in x to avoid the log(0) divergence at the origin
        (the NFW profile is only logarithmically divergent, so this has
        a small, controllable effect -- shrink it and check convergence).
    n_intervals_pow2 : int
        Number of Romberg refinement levels; grid will have 2**n + 1 points.

    Returns
    -------
    frac : float
        M(<x_inner) / M(<c)
    """
    def log_grid_integral(x_lo, x_hi):
        u_lo, u_hi = np.log(x_lo), np.log(x_hi)
        n_points = 2**n_intervals_pow2 + 1
        u = np.linspace(u_lo, u_hi, n_points)
        du = u[1] - u[0]
        x = np.exp(u)
        integrand = x**3 * nfw_shape(x)  # extra factor x from dx = x du
        return romb(integrand, dx=du)

    numerator = log_grid_integral(x_min, x_inner)
    denominator = log_grid_integral(x_min, c)
    return numerator / denominator


def enclosed_fraction_analytic(c, x_inner):
    """Closed-form check using m(x) = ln(1+x) - x/(1+x)."""
    return m_of_c(x_inner) / m_of_c(c)


In [24]:
c_host = 5.2
x_inner = 0.1 * c_host  # e.g. r = 0.1 * r_vir, in units of r_s

f_fiducial = enclosed_fraction_romberg(c_host, x_inner)

for Q_D in [0.8, 1.0, 1.2]:
    c_sub = Q_D * c_host
    x_in = 10**(-1.5) * c_sub  # keep "inner region" at 0.1 * r_vir physically
    x_in = 0.1 * c_sub
    
    frac_romberg = enclosed_fraction_romberg(c_sub, x_in)
    frac_analytic = enclosed_fraction_analytic(c_sub, x_in)

    print(f"Q_D={Q_D:4.2f}  c_sub={c_sub:5.2f}  "
            f"f(analytic)={frac_analytic:.5f}  ")
    
    # print the ratio of fractions compared to Q_D = 1:
    print(f"ratio to Q_D=1: {frac_romberg/f_fiducial:.5f}  ")


Q_D=0.80  c_sub= 4.16  f(analytic)=0.06475  
ratio to Q_D=1: 0.83330  
Q_D=1.00  c_sub= 5.20  f(analytic)=0.07771  
ratio to Q_D=1: 1.00000  
Q_D=1.20  c_sub= 6.24  f(analytic)=0.09005  
ratio to Q_D=1: 1.15890  
